In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import hdf5plugin
from torch.optim.lr_scheduler import ReduceLROnPlateau

class BaselineCNN(nn.Module):
    def __init__(self, num_bins=5, num_classes=1, img_height=480, img_width=640):
        super(BaselineCNN, self).__init__()
        
        self.num_classes = num_classes
        self.img_height = img_height
        self.img_width = img_width
        
        # Simplified but effective feature extractor
        self.feature_extractor = nn.Sequential(
            # First conv block - preserve spatial resolution more
            nn.Conv2d(num_bins, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 16 x 240 x 320
            
            # Second conv block
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 32 x 120 x 160
            
            # Third conv block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 64 x 60 x 80
        )
        
        # Detection head - preserve spatial information
        self.detection_head = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 5, kernel_size=1)  # 5 channels: 4 for bbox + 1 for confidence
        )
        
        # Initialize weights with a better strategy
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Use Xavier/Glorot initialization for better convergence
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        batch_size = x.size(0)
        features = self.feature_extractor(x)
        detection_maps = self.detection_head(features)  # B x 5 x H x W
        conf_map = torch.sigmoid(detection_maps[:, 4:5])  # B x 1 x H x W

        B, _, H, W = conf_map.shape
        flat_conf = conf_map.view(B, -1)
        max_conf_idx = torch.argmax(flat_conf, dim=1)

        y_indices = max_conf_idx // W
        x_indices = max_conf_idx % W

        scale_x = self.img_width / W
        scale_y = self.img_height / H

        bbox_preds = []
        conf_preds = []

        for b in range(B):
            y_idx = y_indices[b]
            x_idx = x_indices[b]
            raw_bbox = detection_maps[b, :4, y_idx, x_idx]

            cx = (x_idx.float() + 0.5) / W
            cy = (y_idx.float() + 0.5) / H
            dx = torch.tanh(raw_bbox[0]) * 0.5
            dy = torch.tanh(raw_bbox[1]) * 0.5

            x_center = torch.clamp(cx + dx, 0, 1)
            y_center = torch.clamp(cy + dy, 0, 1)
            w = torch.sigmoid(raw_bbox[2])
            h = torch.sigmoid(raw_bbox[3])

            # Return normalized coordinates
            bbox_pred = torch.stack([x_center, y_center, w, h])
            bbox_preds.append(bbox_pred)

            conf_pred = conf_map[b, 0, y_idx, x_idx]
            conf_preds.append(conf_pred)

        bbox_preds = torch.stack(bbox_preds)
        conf_preds = torch.stack(conf_preds).unsqueeze(1)

        return bbox_preds, conf_preds, detection_maps

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialDetectionLoss(nn.Module):
    """
    Combined loss function for spatial detection approach
    """
    def __init__(self, img_width=640, img_height=480, lambda_coord=10.0, lambda_noobj=0.1):
        super(SpatialDetectionLoss, self).__init__()
        self.img_width = img_width
        self.img_height = img_height
        self.lambda_coord = lambda_coord  # Weight for bbox coordinate loss
        self.lambda_noobj = lambda_noobj  # Weight for no-object confidence loss
        self.mse_loss = nn.MSELoss(reduction='mean')
        self.bce_loss = nn.BCELoss(reduction='mean')
        

    def forward(self, predictions, targets):
        """
        Compute loss for spatial detection model with normalized coordinates and IoU-based confidence.

        Args:
            predictions: Tuple of (bbox_pred, conf_pred, detection_maps)
                bbox_pred: Tensor of shape (B, 4) with [x, y, w, h] in absolute coordinates
                conf_pred: Tensor of shape (B, 1) with confidence scores
                detection_maps: Tensor of shape (B, 5, H, W) with detection maps
            targets: Tensor of shape (B, 4) with ground truth boxes in absolute coordinates

        Returns:
            total_loss: Combined loss value
            loss_components: Dict with individual loss components
        """
        bbox_pred, conf_pred, detection_maps = predictions
        batch_size = bbox_pred.size(0)

        # Normalize ground truth bounding boxes to [0,1]
        norm_targets = targets / torch.tensor([self.img_width, self.img_height, self.img_width, self.img_height], 
                                              device=targets.device)

        # Normalize predicted bounding boxes to [0,1] for loss calculation
        norm_pred = bbox_pred / torch.tensor([self.img_width, self.img_height, self.img_width, self.img_height], 
                                             device=bbox_pred.device)

        # 1. Coordinate loss (xy and wh)
        xy_loss = self.mse_loss(norm_pred[:, :2], norm_targets[:, :2])
        wh_loss = self.mse_loss(norm_pred[:, 2:], norm_targets[:, 2:])
        coord_loss = xy_loss + 1.5 * wh_loss  # Emphasize size loss

        # 2. Confidence map loss
        B, _, H, W = detection_maps.shape
        confidence_targets = torch.zeros_like(detection_maps[:, 4])  # Confidence heatmap

        for b in range(batch_size):
            x, y, w, h = norm_targets[b]  # Normalized GT coordinates

            # Convert to feature map coordinates
            fm_x = x * W
            fm_y = y * H
            fm_w = w * W
            fm_h = h * H

            # Center of bounding box
            fm_cx = fm_x + fm_w / 2
            fm_cy = fm_y + fm_h / 2

            # Gaussian sigma based on bbox size
            sigma_x = max(W * 2, 5.0)
            sigma_y = max(H * 2, 5.0)

            # Create Gaussian confidence map
            x_coords = torch.arange(0, W, device=targets.device).float()
            y_coords = torch.arange(0, H, device=targets.device).float()
            y_grid, x_grid = torch.meshgrid(y_coords, x_coords, indexing='ij')

            gaussian = torch.exp(
                -0.5 * (((x_grid - fm_cx) / sigma_x) ** 2 + ((y_grid - fm_cy) / sigma_y) ** 2)
            )

            # Apply a threshold to keep meaningful values
            gaussian = torch.where(gaussian > 0.05, gaussian, torch.zeros_like(gaussian))

            # Assign confidence target
            confidence_targets[b] = gaussian

        # 3. Confidence loss (Binary Cross-Entropy)
        conf_map_pred = torch.sigmoid(detection_maps[:, 4])

        # Positive samples (where confidence target > 0.5)
        pos_mask = confidence_targets > 0.5
        pos_loss = F.binary_cross_entropy_with_logits(
            detection_maps[:, 4][pos_mask], 
            confidence_targets[pos_mask],
            reduction='mean'
        ) if pos_mask.sum() > 0 else torch.tensor(0.0, device=targets.device)

        # Hard negative mining
        neg_mask = confidence_targets <= 0.1
        k = 3 * pos_mask.sum().item()
        k = max(k, batch_size)  # Ensure at least batch_size negatives
        k = min(k, neg_mask.sum().item())  # Don't exceed available negatives

        if k > 0 and neg_mask.sum() > 0:
            neg_conf = detection_maps[:, 4][neg_mask]
            neg_sorted, _ = torch.sort(neg_conf, descending=True)
            threshold = neg_sorted[min(k, len(neg_sorted) - 1)]
            hard_neg_mask = (detection_maps[:, 4] > threshold) & neg_mask

            neg_loss = F.binary_cross_entropy_with_logits(
                detection_maps[:, 4][hard_neg_mask],
                confidence_targets[hard_neg_mask],
                reduction='mean'
            )
        else:
            neg_loss = torch.tensor(0.0, device=targets.device)

        # 4. IoU-based direct confidence loss
        ious = torch.tensor([calculate_iou(pred * torch.tensor([self.img_width, self.img_height, self.img_width, self.img_height]),
                                           gt) 
                            for pred, gt in zip(bbox_pred.cpu(), targets.cpu())], device=conf_pred.device)

        direct_conf_loss = F.binary_cross_entropy(conf_pred.squeeze(), ious)

        # 5. Final loss combination
        conf_loss = pos_loss + self.lambda_noobj * neg_loss + direct_conf_loss
        total_loss = self.lambda_coord * coord_loss + conf_loss

        # 6. Return loss components for debugging
        loss_components = {
            'xy_loss': xy_loss.item(),
            'wh_loss': wh_loss.item(),
            'coord_loss': coord_loss.item(),
            'conf_pos_loss': pos_loss.item() if not isinstance(pos_loss, float) else pos_loss,
            'conf_neg_loss': neg_loss.item() if not isinstance(neg_loss, float) else neg_loss,
            'direct_conf_loss': direct_conf_loss.item(),
            'total_loss': total_loss.item()
        }

        return total_loss, loss_components

In [3]:
import torch
import numpy as np
import h5py
from torch.utils.data import Dataset
import os
from tqdm import tqdm
import random
from torch.utils.data.dataloader import default_collate

def create_voxel_grid(events, height=480, width=640, num_bins=5):
    """
    Convert event stream into a voxel grid representation - optimized version.
    """
    # Early return if no events
    if len(events["t"]) == 0:
        return torch.zeros((num_bins, height, width), dtype=torch.float32)
    
    voxel_grid = np.zeros((num_bins, height, width), dtype=np.float32)
    
    t_min, t_max = events["t"].min(), events["t"].max()
    t_range = t_max - t_min + 1e-6  # Prevent division by zero
    
    # Vectorized bin assignment
    bin_indices = np.clip(((events["t"] - t_min) / t_range * num_bins).astype(np.int32), 0, num_bins - 1)
    
    # Filter coordinates that are out of bounds (vectorized)
    valid_indices = (events["x"] >= 0) & (events["x"] < width) & (events["y"] >= 0) & (events["y"] < height)
    x = events["x"][valid_indices]
    y = events["y"][valid_indices]
    b = bin_indices[valid_indices]
    p = events["p"][valid_indices]
    
    # Use numpy's advanced indexing for faster assignment
    for i in range(len(x)):
        voxel_grid[b[i], y[i], x[i]] += 1 if p[i] > 0 else -1
    
    # Quick normalization
    if np.max(np.abs(voxel_grid)) > 0:
        voxel_grid = voxel_grid / np.max(np.abs(voxel_grid))
    
    return torch.tensor(voxel_grid, dtype=torch.float32)

class PrecomputedEventDataset(Dataset):
    """
    Optimized dataset that precomputes voxel grids during initialization
    to avoid repeated computations during training.
    """
    def __init__(self, event_files, label_files, num_bins=5, time_window=10000, 
                 transform=None, frame_step=50, max_samples_per_file=300,
                 precompute=True, cache_dir='./voxel_cache'):
        """
        Dataset for event-based object detection with precomputation.
        
        Args:
            event_files: List of paths to event files
            label_files: List of paths to label files
            num_bins: Number of time bins for voxel grid
            time_window: Time window around each label (in microseconds)
            transform: Optional transforms to apply
            frame_step: Number of frames to skip (higher = faster training)
            max_samples_per_file: Maximum number of samples to take from each file
            precompute: Whether to precompute voxel grids
            cache_dir: Directory to cache precomputed voxel grids (None = no disk caching)
        """
        self.event_files = event_files
        self.label_files = label_files
        self.num_bins = num_bins
        self.time_window = time_window
        self.transform = transform
        self.max_events = 10000  # Reduced max events for faster processing
        self.min_events = 100
        self.frame_step = frame_step
        self.max_samples_per_file = max_samples_per_file
        self.precompute = precompute
        self.cache_dir = cache_dir
        self.img_width = 640
        self.img_height = 480
        
        if self.cache_dir and not os.path.exists(self.cache_dir):
            os.makedirs(self.cache_dir)
        
        # Load labels and correct timestamps
        self.all_samples = []
        
        print("Loading and processing data...")
        for file_idx, (event_file, label_file) in enumerate(tqdm(zip(event_files, label_files), total=len(event_files))):
            try:
                # Load labels
                labels = np.load(label_file, allow_pickle=True)
                
                # Get t_offset
                with h5py.File(event_file, "r", swmr=True) as f:
                    t_offset = f.get("t_offset", 0)[()]
                    # Get all timestamps at once
                    events_t = f["events/t"][:]
                
                # Correct timestamps
                labels["t"] -= t_offset
                
                # Select samples with frame_step
                if self.max_samples_per_file:
                    indices = list(range(0, len(labels), self.frame_step))
                    if len(indices) > self.max_samples_per_file:
                        indices = random.sample(indices, self.max_samples_per_file)
                else:
                    indices = range(0, len(labels), self.frame_step)
                
                # Filter and add valid samples
                for label_idx in indices:
                    if label_idx >= len(labels):
                        continue
                        
                    label = labels[label_idx]
                    timestamp = label["t"]
                    
                    # Find events in time window
                    event_indices = np.where(
                        (events_t >= timestamp - self.time_window) &
                        (events_t <= timestamp + self.time_window)
                    )[0]
                    
                    # Filter by event count
                    if len(event_indices) < self.min_events:
                        continue
                    
                    # Limit max events by random sampling for memory efficiency
                    if len(event_indices) > self.max_events:
                        event_indices = np.random.choice(event_indices, self.max_events, replace=False)
                    
                    # Verify bounding box is within image bounds
                    x, y, w, h = label["x"], label["y"], label["w"], label["h"]
                    if (x < 0 or y < 0 or x + w >= self.img_width or y + h >= self.img_height or
                        w <= 0 or h <= 0 or w > self.img_width or h > self.img_height):
                        continue  # Skip invalid bounding boxes
                    
                    # Store sample info
                    self.all_samples.append({
                        'file_idx': file_idx,
                        'event_file': event_file,
                        'label': {k: label[k] for k in ['x', 'y', 'w', 'h', 'class_id']},
                        'timestamp': timestamp,
                        'event_indices': event_indices
                    })
            except Exception as e:
                print(f"Error processing file {event_file}: {e}")
        
        print(f"Total valid samples: {len(self.all_samples)}")
        
        # Precompute voxel grids if requested
        self.voxel_grids = {}
        if self.precompute:
            self._precompute_voxel_grids()
    
    def _precompute_voxel_grids(self):
        """Precompute voxel grids to avoid repeated computation during training."""
        print("Precomputing voxel grids...")
        
        for idx in tqdm(range(len(self.all_samples))):
            # Check if we have a cached version on disk
            if self.cache_dir:
                cache_file = os.path.join(self.cache_dir, f"voxel_grid_{idx}.pt")
                if os.path.exists(cache_file):
                    self.voxel_grids[idx] = cache_file
                    continue
            
            # Otherwise compute and store
            sample = self.all_samples[idx]
            event_file = sample['event_file']
            event_indices = sample['event_indices']
            
            try:
                with h5py.File(event_file, "r", swmr=True) as f:
                    event_data = {
                        key: f["events/" + key][:][event_indices] 
                        for key in ["t", "x", "y", "p"]
                    }
                
                voxel_grid = create_voxel_grid(event_data, num_bins=self.num_bins)
                
                if self.cache_dir:
                    # Save to disk
                    cache_file = os.path.join(self.cache_dir, f"voxel_grid_{idx}.pt")
                    torch.save(voxel_grid, cache_file)
                    self.voxel_grids[idx] = cache_file
                else:
                    # Keep in memory
                    self.voxel_grids[idx] = voxel_grid
                    
            except Exception as e:
                print(f"Error precomputing voxel grid for sample {idx}: {e}")
                # Create an empty grid as fallback
                self.voxel_grids[idx] = torch.zeros((self.num_bins, self.img_height, self.img_width), dtype=torch.float32)
    
    def clear_cache(self):
        """Clear the disk cache to free up space."""
        if self.cache_dir and os.path.exists(self.cache_dir):
            print(f"Clearing cache directory: {self.cache_dir}")
            for filename in os.listdir(self.cache_dir):
                if filename.startswith("voxel_grid_") and filename.endswith(".pt"):
                    os.remove(os.path.join(self.cache_dir, filename))
            print("Cache cleared.")
    
    def __len__(self):
        return len(self.all_samples)
    
    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        
        # Get voxel grid (either from memory, disk cache, or compute on-the-fly)
        if self.precompute:
            if isinstance(self.voxel_grids[idx], str):
                # Load from disk cache
                voxel_grid = torch.load(self.voxel_grids[idx])
            else:
                # Get from memory
                voxel_grid = self.voxel_grids[idx]
        else:
            # Compute on-the-fly
            event_file = sample['event_file']
            event_indices = sample['event_indices']
            
            try:
                with h5py.File(event_file, "r", swmr=True) as f:
                    event_data = {
                        key: f["events/" + key][:][event_indices] 
                        for key in ["t", "x", "y", "p"]
                    }
                
                voxel_grid = create_voxel_grid(event_data, num_bins=self.num_bins)
                
            except Exception as e:
                print(f"Error computing voxel grid for sample {idx}: {e}")
                voxel_grid = torch.zeros((self.num_bins, self.img_height, self.img_width), dtype=torch.float32)
        
        # Extract label in absolute coordinates
        label = sample['label']
        bbox = torch.tensor([label["x"], label["y"], label["w"], label["h"]], dtype=torch.float32)
        
        # Apply transforms if any
        if self.transform:
            voxel_grid = self.transform(voxel_grid)
        
        return voxel_grid, bbox, label["class_id"]

# Custom collate function to handle errors in batch
def collate_fn(batch):
    """Custom collate function that skips None values in the batch."""
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)

In [6]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import os
import time
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

# Import model, loss and dataset (assuming these files are saved in the same directory)

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Visualization function
def visualize_voxel_grid(voxel_grid, labels=None, save_path=None, detection_map=None):
    if len(voxel_grid.shape) == 4:
        # If batched, take the first sample
        voxel_grid = voxel_grid[0]
    
    # Create a figure with potentially multiple subplots
    fig = plt.figure(figsize=(18, 6))
    
    # Plot 1: Event frame with bounding boxes
    plt.subplot(1, 2 if detection_map is not None else 1, 1)
    
    # Aggregate across time bins for visualization
    event_frame = voxel_grid.sum(dim=0).numpy()
    
    # Normalize for better visualization
    event_frame = (event_frame - event_frame.min()) / (event_frame.max() - event_frame.min() + 1e-6)
    
    plt.imshow(event_frame, cmap="gray")
    
    # Overlay bounding boxes if provided
    if labels is not None:
        for i, det in enumerate(labels):
            color = 'red' if i == 0 else 'lime'  # Ground truth in red, prediction in lime
            label_text = "Ground Truth" if i == 0 else "Prediction"
            
            x, y, w, h = det.tolist()
            rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2)
            plt.gca().add_patch(rect)
            plt.text(x, y - 5, label_text, color=color, fontsize=10,
                     bbox=dict(facecolor="black", alpha=0.5))
    
    plt.title("Event Frame with Bounding Boxes")
    plt.axis("off")
    
    # Plot 2: Confidence map
    if detection_map is not None:
        plt.subplot(1, 2, 2)
        
        # Get confidence map
        conf_map = torch.sigmoid(detection_map[4]).cpu().numpy()
        
        # Display with the Plasma colormap
        plt.imshow(conf_map, cmap="plasma", vmin=0, vmax=1)
        plt.colorbar(label="Confidence")
        plt.title("Detection Confidence Map")
        plt.axis("off")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

# IoU calculation
def calculate_iou(box1, box2):
    # Convert to [x1, y1, x2, y2] format
    box1_x1, box1_y1 = box1[0], box1[1]
    box1_x2, box1_y2 = box1[0] + box1[2], box1[1] + box1[3]
    
    box2_x1, box2_y1 = box2[0], box2[1]
    box2_x2, box2_y2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Calculate intersection area
    x_left = max(box1_x1, box2_x1)
    y_top = max(box1_y1, box2_y1)
    x_right = min(box1_x2, box2_x2)
    y_bottom = min(box1_y2, box2_y2)
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    
    # Calculate union area
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - intersection_area
    
    if union_area <= 0:
        return 0.0
        
    return intersection_area / union_area

# Compute metrics
def calculate_metrics(pred_boxes, gt_boxes, iou_thresholds=[0.5]):
    metrics = {}
    batch_size = pred_boxes.size(0)
    
    # Calculate IoU for each pair of boxes
    ious = torch.zeros(batch_size)
    for i in range(batch_size):
        # Calculate IoU between prediction and ground truth
        iou = calculate_iou(pred_boxes[i].cpu(), gt_boxes[i].cpu())
        ious[i] = iou
    
    # Calculate AP at different IoU thresholds
    for threshold in iou_thresholds:
        matches = (ious > threshold).float()
        precision = matches.mean().item()
        metrics[f'AP@{threshold}'] = precision
    
    # Calculate mean IoU
    metrics['Mean_IoU'] = ious.mean().item()
    
    return metrics

# Debug predictions
def debug_predictions(model, dataloader, device, num_samples=3):
    model.eval()
    examples = []
    
    with torch.no_grad():
        for batch in dataloader:
            if batch is None or len(examples) >= num_samples:
                continue
                
            voxel_grids, bboxes, _ = batch
            voxel_grids = voxel_grids.to(device)
            
            # Get predictions
            bbox_pred, conf_pred, detection_maps = model(voxel_grids)
            
            # Save some examples
            for i in range(min(len(voxel_grids), num_samples - len(examples))):
                examples.append({
                    'voxel_grid': voxel_grids[i].cpu(),
                    'gt_bbox': bboxes[i].cpu(),
                    'pred_bbox': bbox_pred[i].cpu(),
                    'confidence': conf_pred[i].item(),
                    'detection_map': detection_maps[i].cpu()
                })
                
            if len(examples) >= num_samples:
                break
    
    return examples

# Train one epoch
def train_one_epoch(model, dataloader, criterion, optimizer, device, clip_grad=None):
    model.train()
    total_loss = 0
    loss_components = {
        'xy_loss': 0,
        'wh_loss': 0,
        'coord_loss': 0, 
        'conf_pos_loss': 0, 
        'conf_neg_loss': 0, 
        'direct_conf_loss': 0,
        'total_loss': 0
    }
    batches_processed = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for i, batch in enumerate(progress_bar):
        if batch is None:  # Skip bad batches
            continue
            
        voxel_grids, bboxes, _ = batch
        
        # Move data to device
        voxel_grids = voxel_grids.to(device)
        bboxes = bboxes.to(device)
        
        # Forward pass
        predictions = model(voxel_grids)
        
        # Calculate loss
        loss, components = criterion(predictions, bboxes)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            
        optimizer.step()
        
        # Update progress bar
        total_loss += loss.item()
        batches_processed += 1
        
        # Update loss components
        for k, v in components.items():
            loss_components[k] += v
        
        # Show current loss in progress bar
        progress_bar.set_postfix({"loss": total_loss / (batches_processed)})
        
        # Debug training periodically
        if i % 10 == 0 and i > 0:
            with torch.no_grad():
                # Print first prediction vs ground truth
                if len(predictions[0]) > 0:
                    bbox_pred, conf_pred, _ = predictions
                    print(f"\nBatch {i} - Sample 0:")
                    print(f"  GT: {bboxes[0].cpu().numpy()}")
                    print(f"  Pred: {bbox_pred[0].cpu().numpy()}")
                    print(f"  Confidence: {conf_pred[0].item():.4f}")
                    print(f"  IoU: {calculate_iou(bbox_pred[0].cpu(), bboxes[0].cpu()):.4f}")
    
    # Average loss components
    if batches_processed > 0:
        for k in loss_components:
            loss_components[k] /= batches_processed
    
    return total_loss / batches_processed if batches_processed > 0 else float('inf'), loss_components

# Evaluate
def evaluate(model, dataloader, device, iou_thresholds=[0.5]):
    model.eval()
    all_metrics = {f'AP@{threshold}': 0 for threshold in iou_thresholds}
    all_metrics['Mean_IoU'] = 0
    batches_processed = 0
    
    # Debug predictions
    debug_samples = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            if batch is None:  # Skip bad batches
                continue
                
            voxel_grids, bboxes, _ = batch
            
            # Move data to device
            voxel_grids = voxel_grids.to(device)
            bboxes = bboxes.to(device)
            
            # Forward pass
            bbox_pred, conf_pred, _ = model(voxel_grids)
            
            # Store some predictions for debugging
            if len(debug_samples) < 5:
                for i in range(min(2, len(bbox_pred))):
                    debug_samples.append({
                        'gt': bboxes[i].cpu().numpy(),
                        'pred': bbox_pred[i].cpu().numpy(),
                        'conf': conf_pred[i].item()
                    })
            
            # Calculate metrics
            metrics = calculate_metrics(bbox_pred, bboxes, iou_thresholds)
            
            # Accumulate metrics
            for key, value in metrics.items():
                all_metrics[key] += value
            
            batches_processed += 1
    
    # Print debug samples
    print("\n=== Debugging Evaluation Predictions ===")
    for i, sample in enumerate(debug_samples):
        print(f"Sample {i+1} - Ground Truth: {sample['gt']}")
        print(f"Sample {i+1} - Prediction  : {sample['pred']} (Conf: {sample['conf']:.4f})")
    
    # Average metrics
    if batches_processed > 0:
        for key in all_metrics:
            all_metrics[key] /= batches_processed
    
    return all_metrics

# Test IoU calculation
def test_iou():
    # Create two boxes with known IoU
    box1 = torch.tensor([100.0, 100.0, 50.0, 50.0])  # x, y, w, h
    box2 = torch.tensor([125.0, 125.0, 50.0, 50.0])  # This should have IoU of ~0.14
    
    iou = calculate_iou(box1, box2)
    print(f"\n=== Testing IoU Computation ===")
    print(f"Expected IoU ≈ 0.14, Computed IoU: {iou}")

# Function to clear cache
def clear_cache(cache_dir='./voxel_cache'):
    if os.path.exists(cache_dir):
        print(f"Clearing cache directory: {cache_dir}")
        for filename in os.listdir(cache_dir):
            if filename.startswith("voxel_grid_") and filename.endswith(".pt"):
                os.remove(os.path.join(cache_dir, filename))
        print("Cache cleared.")
    else:
        print(f"Cache directory {cache_dir} does not exist.")

# Configuration
config = {
    'num_bins': 5,
    'time_window': 20000,
    'batch_size': 8,
    'num_epochs': 30,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'dataset_path': '../../Datasets/DSEC_Detection/dsec-det/',
    'save_dir': './results',
    'evaluate_every': 1,
    'frame_step': 50,
    'max_samples_per_file': 100,  
    'clip_grad': 10.0,
    'cache_dir': './voxel_cache',
    'precompute': True,
    'lambda_coord': 10.0,
    'lambda_noobj': 1.0,
    'clear_cache': False  # Set to True to clear cache before training
}

# Main training function
def train_model(config):
    # Set seed for reproducibility
    set_seed(42)
    
    print(f"Using device: {config['device']}")
    
    # Create directories
    os.makedirs(config['save_dir'], exist_ok=True)
    if config['cache_dir']:
        os.makedirs(config['cache_dir'], exist_ok=True)
    
    # Clear cache if requested
    if config['clear_cache']:
        clear_cache(config['cache_dir'])
    
    # Prepare dataset paths
    dataset_path = config['dataset_path']
    train_event_dir = os.path.join(dataset_path, 'train_events/train/')
    train_label_dir = os.path.join(dataset_path, 'train_object_detections/train/')
    
    # Find sequences
    sequences = [d for d in os.listdir(train_event_dir) if os.path.isdir(os.path.join(train_event_dir, d))]
    print(f"Found {len(sequences)} sequences.")
    
    # Get file paths
    event_files = []
    label_files = []
    
    for seq in sequences:
        event_file = os.path.join(train_event_dir, seq, 'events/left/events.h5')
        label_file = os.path.join(train_label_dir, seq, 'object_detections/left/tracks.npy')
        
        if os.path.exists(event_file) and os.path.exists(label_file):
            event_files.append(event_file)
            label_files.append(label_file)
            print(f"✓ Found: {seq}")
    
    print(f"Found {len(event_files)} valid sequence pairs.")
    
    # Test IoU calculation
    test_iou()
    
    # Create dataset
    dataset = PrecomputedEventDataset(
        event_files=event_files,
        label_files=label_files,
        num_bins=config['num_bins'],
        time_window=config['time_window'],
        frame_step=config['frame_step'],
        max_samples_per_file=config['max_samples_per_file'],
        precompute=config['precompute'],
        cache_dir=config['cache_dir']
    )
    
    # Split dataset
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], 
                                              generator=torch.Generator().manual_seed(42))
    
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=16,
        pin_memory=True,
        collate_fn=collate_fn,
        persistent_workers=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=16,
        pin_memory=True,
        collate_fn=collate_fn,
        persistent_workers=True
    )
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    
    # Create model
    model = BaselineCNN(
        num_bins=config['num_bins'], 
        img_height=480, 
        img_width=640
    ).to(config['device'])
    
    # Custom loss function
    criterion = SpatialDetectionLoss(
        img_width=640, 
        img_height=480,
        lambda_coord=config['lambda_coord'],
        lambda_noobj=config['lambda_noobj']
    )
    
    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    # Adjusted
    
    # Training loop
    best_ap = 0
    best_epoch = -1
    best_iou = 0
    start_time = time.time()
    
    print("\n=== Starting Training ===")
    
    for epoch in range(config['num_epochs']):
        epoch_start = time.time()
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        
        # Train
        train_loss, loss_components = train_one_epoch(
            model, train_dataloader, criterion, optimizer, 
            config['device'], clip_grad=config['clip_grad']
        )
        
        epoch_time = time.time() - epoch_start
        print(f"Train Loss: {train_loss:.4f} (Time: {epoch_time:.2f}s)")
        print("Loss Components:")
        for k, v in loss_components.items():
            print(f"  {k}: {v:.4f}")
        
        # Update learning rate
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current learning rate: {current_lr:.6f}")
        
        # Evaluate
        if (epoch + 1) % config['evaluate_every'] == 0 or epoch == config['num_epochs'] - 1:
            metrics = evaluate(model, val_dataloader, config['device'])
            
            print("Validation Metrics:")
            for key, value in metrics.items():
                print(f"  {key}: {value:.4f}")
            
            # Save best AP model
            if metrics['AP@0.5'] > best_ap:
                best_ap = metrics['AP@0.5']
                best_epoch = epoch
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'metrics': metrics,
                }, os.path.join(config['save_dir'], 'best_ap_model.pth'))
                print(f"Saved new best AP model with AP@0.5: {best_ap:.4f}")
            
            # Also save best IoU model
            if metrics['Mean_IoU'] > best_iou:
                best_iou = metrics['Mean_IoU']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'metrics': metrics,
                }, os.path.join(config['save_dir'], 'best_iou_model.pth'))
                print(f"Saved new best IoU model with Mean_IoU: {best_iou:.4f}")
            
            # Get some debug examples
            if epoch % 3 == 0 or epoch == config['num_epochs'] - 1:
                examples = debug_predictions(model, val_dataloader, config['device'])
                
                # Visualize examples
                for i, example in enumerate(examples):
                    fig_path = os.path.join(config['save_dir'], f'debug_epoch{epoch+1}_ex{i+1}.png')
                    visualize_voxel_grid(
                        example['voxel_grid'],
                        labels=[example['gt_bbox'], example['pred_bbox']],
                        save_path=fig_path,
                        detection_map=example['detection_map']
                    )
                    print(f"Saved debug visualization to {fig_path}")
    
    total_time = time.time() - start_time
    print(f"\nTraining complete! Total time: {total_time:.2f}s")
    print(f"Best AP@0.5: {best_ap:.4f} at epoch {best_epoch+1}")
    print(f"Best Mean IoU: {best_iou:.4f}")
    
    # Return the trained model and metrics
    return model, {'best_ap': best_ap, 'best_iou': best_iou, 'best_epoch': best_epoch}

              
    # To run the training directly (uncomment when needed)
if __name__ == "__main__":
     model, metrics = train_model(config)


Using device: cpu
Found 6 sequences.
✓ Found: zurich_city_17_a
✓ Found: zurich_city_20_a
✓ Found: zurich_city_18_a
✓ Found: zurich_city_16_a
✓ Found: zurich_city_21_a
✓ Found: zurich_city_19_a
Found 6 valid sequence pairs.

=== Testing IoU Computation ===
Expected IoU ≈ 0.14, Computed IoU: 0.1428571492433548
Loading and processing data...


100%|██████████| 6/6 [14:35<00:00, 145.85s/it]


Total valid samples: 421
Precomputing voxel grids...


100%|██████████| 421/421 [00:00<00:00, 79943.95it/s]


Train dataset size: 336
Validation dataset size: 85

=== Starting Training ===

Epoch 1/30


...


Batch 30 - Sample 0:
  GT: [109. 204.   9.  18.]
  Pred: [0.52064514 0.6612997  0.81898254 0.8344738 ]
  Confidence: 0.4888
  IoU: 0.0000


Training: 100%|██████████| 42/42 [00:06<00:00,  6.20it/s, loss=3.7] 



Batch 40 - Sample 0:
  GT: [227. 215.  19.  14.]
  Pred: [0.8559892  1.         0.99826956 0.99863595]
  Confidence: 0.4892
  IoU: 0.0000
Train Loss: 3.6953 (Time: 6.78s)
Loss Components:
  xy_loss: 0.2215
  wh_loss: 0.0060
  coord_loss: 0.2304
  conf_pos_loss: 0.7209
  conf_neg_loss: 0.0000
  direct_conf_loss: 0.6707
  total_loss: 3.6953
Current learning rate: 0.000250


Evaluating: 100%|██████████| 11/11 [00:00<00:00, 18.61it/s]



=== Debugging Evaluation Predictions ===
Sample 1 - Ground Truth: [502. 163.  12.  11.]
Sample 1 - Prediction  : [1.         0.7242596  0.9977806  0.99824476] (Conf: 0.4886)
Sample 2 - Ground Truth: [329. 215.  34.  30.]
Sample 2 - Prediction  : [0.93171924 1.         0.96725297 0.9676014 ] (Conf: 0.4887)
Sample 3 - Ground Truth: [371. 213.  14.  11.]
Sample 3 - Prediction  : [1.         0.5111705  0.87535065 0.87844944] (Conf: 0.4887)
Sample 4 - Ground Truth: [579. 223.  10.  23.]
Sample 4 - Prediction  : [0.90571797 0.7572934  0.99708515 0.9975643 ] (Conf: 0.4886)
Sample 5 - Ground Truth: [ 45. 218.   9.  20.]
Sample 5 - Prediction  : [0.7555666 0.7403368 0.9964497 0.996958 ] (Conf: 0.4887)
Sample 6 - Ground Truth: [392. 215.  18.  11.]
Sample 6 - Prediction  : [0.7308696  1.         0.99779886 0.9982658 ] (Conf: 0.4886)
Validation Metrics:
  AP@0.5: 0.0000
  Mean_IoU: 0.0000
Saved debug visualization to ./results/debug_epoch30_ex1.png
Saved debug visualization to ./results/debug_ep

In [ ]:
#fix data lading logic <- Done
#Use YOLO instead of manualy iou boxing, simplify process.
# Please finish this base comparison so i can move on 
# mac file count = 200, max sequence also set, look at example images and adjust frames

In [ ]:
#finish and implement CREST by thursday, porbably should compress data as voxel grid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Function to load events from an HDF5 file within a time window
def load_events(event_file, time_window):
    with h5py.File(event_file, 'r') as f:
        t = f['events/t'][:]  # Timestamps
        x = f['events/x'][:]  # X coordinates
        y = f['events/y'][:]  # Y coordinates
        p = f['events/p'][:]  # Polarity
        
        # Take events from the first time_window microseconds
        mask = t < t[0] + time_window
        return {'t': t[mask], 'x': x[mask], 'y': y[mask], 'p': p[mask]}

# Function to create a 2D image from events
def create_event_image(events, height=480, width=640):
    img = np.zeros((height, width), dtype=np.float32)
    for event in zip(events['x'], events['y'], events['p']):
        if 0 <= event[1] < height and 0 <= event[0] < width:
            img[event[1], event[0]] += 1 if event[2] > 0 else -1
    # Normalize for better visualization
    if np.max(np.abs(img)) > 0:
        img /= np.max(np.abs(img))
    return img

# Function to visualize the event image with a bounding box
def visualize_with_bbox(event_image, bbox):
    plt.imshow(event_image, cmap='gray')
    x, y, w, h = bbox  # Unpack bounding box coordinates
    rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2)
    plt.gca().add_patch(rect)
    plt.savefig('check_label.png')  # Save to file
    plt.close()  # Close the plot to avoid display

# Example usage
event_file = '../../Datasets/DSEC_Detection/dsec-det/train_events/train/zurich_city_18_a/events/left/events.h5'
label_file = '../../Datasets/DSEC_Detection/dsec-det/train_object_detections/train/zurich_city_18_a/object_detections/left/tracks.npy'
time_window = 30000  # Time window in microseconds

# Load labels (assuming a NumPy array with 'x', 'y', 'w', 'h' for each box)
labels = np.load(label_file, allow_pickle=True)
bbox = labels[0]['x'], labels[0]['y'], labels[0]['w'], labels[0]['h']  # Use the first label

# Load events and create the event image
events = load_events(event_file, time_window)
event_image = create_event_image(events)

# Visualize and save the result
visualize_with_bbox(event_image, bbox)

In [ ]:
print("above completed")

In [11]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/mnt/iusers01/fse-ugpgt01/compsci01/u57436ko/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [20]:
import os
import shutil
import numpy as np
import h5py
import hdf5plugin
from PIL import Image
from ultralytics import YOLO
import torch
import random

# ==========================
# 📂 Directory Setup
# ==========================
cache_dir = os.path.abspath("voxel_cache")  # Ensure absolute path
os.makedirs(cache_dir, exist_ok=True)

# ==========================
# 🔍 Load Event Data & Labels
# ==========================
event_file = "../../Datasets/DSEC_Detection/dsec-det/train_events/train/zurich_city_18_a/events/left/events.h5"
label_file = "../../Datasets/DSEC_Detection/dsec-det/train_object_detections/train/zurich_city_18_a/object_detections/left/tracks.npy"

# Load labels (bounding boxes)
labels = np.load(label_file, allow_pickle=True)

# Open HDF5 file
with h5py.File(event_file, "r") as f:
    event_data = {key: f["events/" + key][:] for key in ["t", "x", "y", "p"]}
    t_offset = f.get("t_offset", 0)[()]

print(f"🕒 Loaded `t_offset`: {t_offset}")

# ==========================
# ⏳ Correct Time Alignment
# ==========================
labels["t"] -= t_offset
print(f"🔍 Unique Tracks Available: {np.unique(labels['track_id'])}")

# ==========================
# 🎯 Process Multiple Time Windows
# ==========================
time_window = 10000  # ±20ms
num_samples = 10  # Number of samples to generate (adjust as needed)

for sample_idx in range(num_samples):
    # Select a random base timestamp
    base_label_timestamp = random.choice(labels)["t"]
    matching_labels = labels[(labels["t"] >= base_label_timestamp - time_window) & 
                            (labels["t"] <= base_label_timestamp + time_window)]

    print(f"Sample {sample_idx} - 🎯 Base Timestamp: {base_label_timestamp}, 🔎 {len(matching_labels)} matching labels")

    # Expand event search window
    event_indices = np.where((event_data["t"] >= base_label_timestamp - time_window) & 
                             (event_data["t"] <= base_label_timestamp + time_window))[0]
    print(f"🔎 Found {len(event_indices)} events")

    if not len(event_indices):
        print("⚠️ No events found. Skipping this sample.")
        continue

    # Generate event frame
    event_frame = np.zeros((480, 640))
    np.add.at(event_frame, (event_data["y"][event_indices], event_data["x"][event_indices]), 1)

    # Normalize for saving as an image
    event_frame = (np.log1p(event_frame) / np.log1p(event_frame.max()) * 255).astype(np.uint8) if event_frame.max() > 0 else event_frame

    # Save the event frame as an image
    image_path = os.path.join(cache_dir, f"sample_{sample_idx}.png")
    Image.fromarray(event_frame).save(image_path)

    # Prepare labels in YOLO format
    label_path = os.path.join(cache_dir, f"sample_{sample_idx}.txt")
    with open(label_path, "w") as f:
        for det in matching_labels:
            x, y, w, h = det["x"], det["y"], det["w"], det["h"]
            class_id = det["class_id"]
            x_center = (x + w / 2) / 640  # Normalize to [0, 1]
            y_center = (y + h / 2) / 480
            width = w / 640
            height = h / 480
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

# ==========================
# 🔍 **Sanity Check: Ensure Images & Labels Exist**
# ==========================
image_files = [f for f in os.listdir(cache_dir) if f.endswith(".png")]
label_files = [f for f in os.listdir(cache_dir) if f.endswith(".txt")]

if len(image_files) == 0 or len(label_files) == 0:
    raise FileNotFoundError(f"⚠️ No images or labels found in {cache_dir}. Check data preprocessing!")

print(f"✅ Found {len(image_files)} images and {len(label_files)} labels in `{cache_dir}`.")

# ==========================
# 📂 Set Up Dataset Configuration
# ==========================
dataset_yaml = f"""
train: {cache_dir}
val: {cache_dir}
nc: 1  # Number of classes (adjust as needed)
names: ['object']  # Class name (adjust as needed)
"""
yaml_path = os.path.join(cache_dir, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(dataset_yaml)

print(f"✅ Dataset YAML created at: {yaml_path}")

# ==========================
# 🧠 Configure and Train YOLO
# ==========================
# Load YOLOv8 model (nano version for simplicity)
model = YOLO("yolov8n.pt")

# Modify the model to accept single-channel input (grayscale images)
model.model.model[0].conv = torch.nn.Conv2d(1, model.model.model[0].conv.out_channels, 
                                            kernel_size=3, stride=1, padding=1)

# ==========================
# 🚀 Train the Model
# ==========================
print("🔄 Starting YOLO training...")
model.train(data=yaml_path, epochs=30, imgsz=640, batch=8)

# ==========================
# 📊 Evaluate the Model
# ==========================
results = model.val()
print("✅ Evaluation Results:")
print(results)

Sample 0 - 🎯 Base Timestamp: 40556957, 🔎 7 matching labels
🔎 Found 206066 events
🔍 Class ID found: 0
🔍 Class ID found: 1
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 5
Sample 1 - 🎯 Base Timestamp: 42505317, 🔎 7 matching labels
🔎 Found 296200 events
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
Sample 2 - 🎯 Base Timestamp: 4857589, 🔎 3 matching labels
🔎 Found 214517 events
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
Sample 3 - 🎯 Base Timestamp: 27307597, 🔎 3 matching labels
🔎 Found 248490 events
🔍 Class ID found: 2
🔍 Class ID found: 2
🔍 Class ID found: 2
Sample 4 - 🎯 Base Timestamp: 49507049, 🔎 5 matching labels
🔎 Found 211182 events
🔍 Class ID found: 0
🔍 Class ID found: 0
🔍 Class ID found: 0
🔍 Class ID found: 2
🔍 Class ID found: 5
Sample 5 - 🎯 Base Timestamp: 46657004, 🔎 6 matching labels
🔎 Found 305755 events
🔍 Class ID foun

In [3]:
import os
import shutil

# ==========================
# 🗑️ Clear Cache Directory
# ==========================
cache_dir = "voxel_cache"

def clear_cache():
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        os.makedirs(cache_dir)
        print("✅ Cache directory cleared and recreated.")
    else:
        os.makedirs(cache_dir)
        print("✅ Cache directory did not exist; created a new one.")

# Run the clear function
clear_cache()

✅ Cache directory cleared and recreated.


In [2]:
import os
import numpy as np
import h5py
from PIL import Image
from ultralytics import YOLO
from multiprocessing import Pool
import hdf5plugin

# --------------------------
# Directory Setup
# --------------------------
cache_dir = "voxel_cache"
train_dir = os.path.join(cache_dir, "train")
val_dir = os.path.join(cache_dir, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Load your event and label data (adjust paths as needed)
event_file = "../../Datasets/DSEC_Detection/dsec-det/train_events/train/zurich_city_18_a/events/left/events.h5"
label_file = "../../Datasets/DSEC_Detection/dsec-det/train_object_detections/train/zurich_city_18_a/object_detections/left/tracks.npy"

labels = np.load(label_file, allow_pickle=True)
with h5py.File(event_file, "r") as f:
    event_data = {key: f["events/" + key][:] for key in ["t", "x", "y", "p"]}
    t_offset = f.get("t_offset", 0)[()]
labels["t"] -= t_offset

# Get unique timestamps from labels
unique_timestamps = np.unique(labels['t'])
time_window = 10000  # ±10ms window for events

# --------------------------
# Function to Process Timestamp
# --------------------------
def process_timestamp(base_label_timestamp):
    # Find labels for this timestamp
    matching_labels = labels[labels['t'] == base_label_timestamp]
    
    # Aggregate events in the time window using histogram2d
    event_indices = np.where((event_data['t'] >= base_label_timestamp - time_window) &
                             (event_data['t'] <= base_label_timestamp + time_window))[0]
    
    if not event_indices.size:
        print(f"[DEBUG] No events found for timestamp: {base_label_timestamp}")
        return None
    
    event_x = event_data['x'][event_indices]
    event_y = event_data['y'][event_indices]
    event_frame, _, _ = np.histogram2d(event_y, event_x, bins=(480, 640), range=[[0, 480], [0, 640]])
    
    # Normalize to 0-255 for image saving
    if event_frame.max() > 0:
        event_frame = (np.log1p(event_frame) / np.log1p(event_frame.max()) * 255).astype(np.uint8)
    
    print(f"[DEBUG] Processed timestamp: {base_label_timestamp} (events: {len(event_indices)})")
    return event_frame, matching_labels

# --------------------------
# Main Processing Block
# --------------------------
if __name__ == '__main__':
    print(f"[DEBUG] Total unique timestamps: {len(unique_timestamps)}")
    batch_size = 50  # Smaller batch size for slower RAM/system
    num_batches = len(unique_timestamps) // batch_size + 1

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(unique_timestamps))
        timestamps_batch = unique_timestamps[start_idx:end_idx]
        
        print(f"[DEBUG] Processing batch {batch_idx + 1}/{num_batches} with {len(timestamps_batch)} timestamps")
        with Pool(processes=2) as pool:
            results = pool.map(process_timestamp, timestamps_batch)
        print(f"[DEBUG] Completed batch {batch_idx + 1}")
        
        # Save results for this batch
        for sample_idx, result in enumerate(results):
            if result is None:
                continue
            event_frame, matching_labels = result
            global_sample_idx = start_idx + sample_idx
            
            # Split into train (80%) and val (20%)
            save_dir = train_dir if global_sample_idx < int(0.8 * len(unique_timestamps)) else val_dir
            image_path = os.path.join(save_dir, f"sample_{global_sample_idx}.png")
            label_path = os.path.join(save_dir, f"sample_{global_sample_idx}.txt")
            
            # Save image
            Image.fromarray(event_frame).convert('RGB').save(image_path)
            print(f"[DEBUG] Saved image: {image_path}")
            
            # Save YOLO labels
            with open(label_path, "w") as f:
                for det in matching_labels:
                    x, y, w, h = det["x"], det["y"], det["w"], det["h"]
                    class_id = det["class_id"]
                    x_center = min(max((x + w / 2) / 640, 0), 1)
                    y_center = min(max((y + h / 2) / 480, 0), 1)
                    width = min(max(w / 640, 0), 1)
                    height = min(max(h / 480, 0), 1)
                    f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
            print(f"[DEBUG] Saved labels: {label_path}")
    
    # Create dataset.yaml for YOLO
    dataset_yaml = f"""
train: {os.path.abspath(train_dir)}
val: {os.path.abspath(val_dir)}
nc: 8
names: ['pedestrian', 'rider', 'car', 'bus', 'truck', 'bicycle', 'motorcycle', 'train']
"""
    yaml_path = os.path.join(cache_dir, "dataset.yaml")
    with open(yaml_path, "w") as f:
        f.write(dataset_yaml)
    print(f"[DEBUG] Created dataset YAML at: {yaml_path}")
    
    # Train YOLO model
    model = YOLO("yolov8n.pt")
    print("[DEBUG] Starting YOLO training...")
    model.train(data=yaml_path, epochs=30, imgsz=640, batch=8)
    
    # Evaluate the model
    results = model.val()
    print("Evaluation Results:", results)


[DEBUG] Total unique timestamps: 1503
[DEBUG] Processing batch 1/31 with 50 timestamps
[DEBUG] Processed timestamp: 355371 (events: 10051)[DEBUG] Processed timestamp: 544 (events: 5087)

[DEBUG] Processed timestamp: 405727 (events: 10023)[DEBUG] Processed timestamp: 50593 (events: 10310)

[DEBUG] Processed timestamp: 101435 (events: 10398)[DEBUG] Processed timestamp: 456028 (events: 10479)

[DEBUG] Processed timestamp: 506319 (events: 10289)[DEBUG] Processed timestamp: 152563 (events: 10134)

[DEBUG] Processed timestamp: 203601 (events: 10180)[DEBUG] Processed timestamp: 556583 (events: 13119)

[DEBUG] Processed timestamp: 254411 (events: 10085)[DEBUG] Processed timestamp: 606797 (events: 25583)

[DEBUG] Processed timestamp: 304959 (events: 10183)[DEBUG] Processed timestamp: 657014 (events: 29636)

[DEBUG] Processed timestamp: 707191 (events: 25561)[DEBUG] Processed timestamp: 1057549 (events: 40539)

[DEBUG] Processed timestamp: 757347 (events: 24539)[DEBUG] Processed timestamp: 11075

[DEBUG] Processed timestamp: 5357581 (events: 91525)[DEBUG] Processed timestamp: 5007541 (events: 156160)

[DEBUG] Processed timestamp: 5407575 (events: 87736)[DEBUG] Processed timestamp: 5057571 (events: 138654)

[DEBUG] Processed timestamp: 5457577 (events: 83636)[DEBUG] Processed timestamp: 5107579 (events: 129287)

[DEBUG] Processed timestamp: 5507581 (events: 84322)[DEBUG] Processed timestamp: 5157577 (events: 125935)

[DEBUG] Processed timestamp: 5557583 (events: 86176)
[DEBUG] Processed timestamp: 5207583 (events: 120534)
[DEBUG] Processed timestamp: 5607585 (events: 89050)[DEBUG] Processed timestamp: 5257579 (events: 109040)

[DEBUG] Processed timestamp: 5657581 (events: 97494)
[DEBUG] Processed timestamp: 5307581 (events: 99446)
[DEBUG] Processed timestamp: 5707603 (events: 113332)
[DEBUG] Processed timestamp: 6057579 (events: 302691)
[DEBUG] Processed timestamp: 5757595 (events: 141184)
[DEBUG] Processed timestamp: 6107573 (events: 322720)
[DEBUG] Processed timestamp: 5807597

[DEBUG] Processed timestamp: 7857593 (events: 313083)[DEBUG] Processed timestamp: 7507595 (events: 346437)

[DEBUG] Processed timestamp: 7907593 (events: 314523)
[DEBUG] Processed timestamp: 7557597 (events: 339443)
[DEBUG] Processed timestamp: 7957595 (events: 311337)
[DEBUG] Processed timestamp: 7607599 (events: 329848)
[DEBUG] Processed timestamp: 8007545 (events: 306648)
[DEBUG] Processed timestamp: 7657599 (events: 317317)
[DEBUG] Processed timestamp: 8057573 (events: 310529)[DEBUG] Processed timestamp: 7707605 (events: 306110)

[DEBUG] Processed timestamp: 7757589 (events: 302183)[DEBUG] Processed timestamp: 8107591 (events: 319760)

[DEBUG] Processed timestamp: 7807591 (events: 307706)
[DEBUG] Processed timestamp: 8157587 (events: 332147)
[DEBUG] Processed timestamp: 8207589 (events: 338127)[DEBUG] Processed timestamp: 8557609 (events: 326620)

[DEBUG] Processed timestamp: 8607603 (events: 324554)[DEBUG] Processed timestamp: 8257591 (events: 344017)

[DEBUG] Processed timestamp:

[DEBUG] Processed timestamp: 10357591 (events: 214388)[DEBUG] Processed timestamp: 10007543 (events: 240927)

[DEBUG] Processed timestamp: 10407597 (events: 202977)[DEBUG] Processed timestamp: 10057575 (events: 232661)

[DEBUG] Processed timestamp: 10457593 (events: 198498)[DEBUG] Processed timestamp: 10107569 (events: 230710)

[DEBUG] Processed timestamp: 10507595 (events: 187685)[DEBUG] Processed timestamp: 10157589 (events: 230549)

[DEBUG] Processed timestamp: 10557595 (events: 177358)[DEBUG] Processed timestamp: 10207591 (events: 226013)

[DEBUG] Processed timestamp: 10607593 (events: 161137)[DEBUG] Processed timestamp: 10257573 (events: 223862)

[DEBUG] Processed timestamp: 10657603 (events: 149635)[DEBUG] Processed timestamp: 10307589 (events: 218423)

[DEBUG] Processed timestamp: 10707603 (events: 132863)[DEBUG] Processed timestamp: 11057581 (events: 105573)

[DEBUG] Processed timestamp: 10757601 (events: 119498)[DEBUG] Processed timestamp: 11107569 (events: 101773)

[DEBUG] Pr

[DEBUG] Processed timestamp: 12507589 (events: 207036)[DEBUG] Processed timestamp: 12857591 (events: 232189)

[DEBUG] Processed timestamp: 12907587 (events: 239754)[DEBUG] Processed timestamp: 12557591 (events: 206889)

[DEBUG] Processed timestamp: 12957589 (events: 245103)[DEBUG] Processed timestamp: 12607585 (events: 213970)

[DEBUG] Processed timestamp: 13007543 (events: 252796)[DEBUG] Processed timestamp: 12657589 (events: 216736)

[DEBUG] Processed timestamp: 13057579 (events: 262308)[DEBUG] Processed timestamp: 12707595 (events: 223468)

[DEBUG] Processed timestamp: 13107569 (events: 262825)[DEBUG] Processed timestamp: 12757587 (events: 224560)

[DEBUG] Processed timestamp: 12807589 (events: 230637)[DEBUG] Processed timestamp: 13157593 (events: 264194)

[DEBUG] Processed timestamp: 13557591 (events: 238804)[DEBUG] Processed timestamp: 13207587 (events: 262157)

[DEBUG] Processed timestamp: 13607593 (events: 248434)[DEBUG] Processed timestamp: 13257589 (events: 256991)

[DEBUG] Pr

...


30 epochs completed in 0.883 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 6.2MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.76 🚀 Python-3.8.20 torch-2.4.1+cu121 CPU (Intel Xeon Gold 6326 2.90GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:06<00:00,  2.88it/s]


                   all        301        852      0.728      0.117      0.209      0.121
            pedestrian        107        164      0.326      0.128      0.111     0.0502
                   car        301        673      0.588       0.34      0.354      0.183
               bicycle         12         12          1          0          0          0
            motorcycle          3          3          1          0      0.369      0.252
Speed: 0.3ms preprocess, 13.3ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/detect/train5
Ultralytics 8.3.76 🚀 Python-3.8.20 torch-2.4.1+cu121 CPU (Intel Xeon Gold 6326 2.90GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /net/scratch2/u57436ko/SNN_Project/notebooks/voxel_cache/val.cache... 301 images, 0 backgrounds, 0 corrupt: 100%|██████████| 301/301 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 38/38 [00:06<00:00,  6.06it/s]


                   all        301        852      0.728      0.117      0.209      0.121
            pedestrian        107        164      0.326      0.128      0.111     0.0502
                   car        301        673      0.588       0.34      0.354      0.183
               bicycle         12         12          1          0          0          0
            motorcycle          3          3          1          0      0.369      0.252
Speed: 0.2ms preprocess, 13.3ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/detect/train52
Evaluation Results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x2b125443cac0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.